# WavLM Embedding Evaluation - Interactive Visualization

This notebook allows for interactive inspection of speaker embeddings across different PC-GITA groups, with gender-aware coloring.

In [ ]:
import os
import sys
import ipywidgets as widgets
from IPython.display import display

# Add root to path if needed
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from embeddings_eval.data_loader import load_embeddings, load_metadata
from embeddings_eval.analyzer import WavLMAnalyzer
from embeddings_eval.visualizer import WavLMVisualizer
from embeddings_eval.constants import ALL_GROUPS

In [ ]:
DATA_DIR = "../datalocal/PC-GITA_v260210_24kHz/speaker_embeddings/wavLM"
META_PATH = "../datalocal/PC-GITA_v260210_24kHz/_metadata/PCGITAtoPD_mapping.csv"

print("Loading metadata...")
metadata = load_metadata(META_PATH)

print(f"Loading embeddings from {DATA_DIR}...")
embeddings = load_embeddings(DATA_DIR, metadata=metadata)
analyzer = WavLMAnalyzer(embeddings)
visualizer = WavLMVisualizer(analyzer)
print(f"Loaded {len(embeddings)} samples across {len(analyzer.speaker_centroids[analyzer.versions[0]])} speakers.")

## Global Centroid Map
The following plot shows all speaker global centroids and their task-specific group centroids. Use this to identify overall clusters and outlier speakers.

In [ ]:
visualizer.plot_centroids(method='pca')

## Detailed Speaker Inspection
Select specific speakers to see their individual sample distributions and how they relate to their group and global centroids.

In [ ]:
def update_plot(method, selected_speakers):
    if not selected_speakers:
        print("Select at least one speaker")
        return
    visualizer.plot_speaker_comparison(list(selected_speakers), method=method.lower())

ver = analyzer.versions[0]
speakers = sorted(list(analyzer.speaker_centroids[ver].keys()))
method_dropdown = widgets.Dropdown(options=['PCA', 't-SNE'], value='PCA', description='Method:')
speaker_select = widgets.SelectMultiple(options=speakers, value=[speakers[0]], description='Speakers:', rows=10)

widgets.interactive(update_plot, method=method_dropdown, selected_speakers=speaker_select)